# Experiment 1: Snapdragon Backend Setup

## Objective

The objective of this experiment is to prepare a Snapdragon-enabled version of `llama.cpp` capable of targeting Qualcomm hardware.

Unlike the CPU workflow, Snapdragon execution requires additional toolchains for Android cross-compilation and Qualcomm hardware support.

This experiment focuses only on building the required binaries.

No model inference is performed in this notebook.

## Expected Outcome

By the end of this experiment, the following should be available:

- Android ARM64 binaries
- Snapdragon-enabled `llama-cli`
- Snapdragon-enabled `llama-server`
- Snapdragon-enabled `llama-bench`
- Qualcomm backend libraries ready for deployment

## Background

The CPU deployment workflow has already been completed successfully.

Current benchmark:

```text
Device      : Samsung Galaxy S26 Ultra
Chipset     : Snapdragon 8 Elite Gen 5
Memory      : 12 GB
Backend     : CPU

Prompt Processing : ~13.6 Tokens/sec
Generation        : ~10 Tokens/sec
```

The next objective is to investigate whether the Qualcomm Snapdragon backend can be used to accelerate inference using the Hexagon NPU.

## Case Study

Before beginning implementation, the following observations were made.

### Observation 1

The latest `llama.cpp` includes official support for Qualcomm Snapdragon platforms.

Supported execution backends include:

- CPU
- Adreno GPU
- Hexagon NPU

---

### Observation 2

The Hexagon backend exposes logical execution devices such as:

```text
HTP0
HTP1
HTP2
...
```

These are runtime execution targets managed by the Qualcomm AI Runtime and do not represent multiple physical devices.

---

### Observation 3

The CPU workflow provides the baseline implementation.

The Snapdragon workflow aims to replace or augment CPU execution using Qualcomm hardware acceleration while preserving the same GGUF deployment pipeline.

---

### Outcome

The first implementation step is to successfully build a Snapdragon-compatible version of `llama.cpp`.


## Experiment Goal

The experiment is considered successful if:

- Snapdragon backend compiles successfully
- Android ARM64 executables are generated
- Qualcomm backend libraries are generated
- Binaries are ready for deployment to the Galaxy S26 Ultra

## Development Environment

### Host

```text
Operating System : Windows 11
Development Tools: Git, Docker Desktop, CMake
```

### Target Device

```text
Device      : Samsung Galaxy S26 Ultra
Chipset     : Snapdragon 8 Elite Gen 5
Memory      : 12 GB
Storage     : 256 GB
Platform    : Android
```

### Reference

This experiment follows the official Snapdragon backend documentation provided by the `llama.cpp` project.

---


## Prerequisites

Before building the Snapdragon backend, verify that the required development tools are available on the host machine.

This experiment uses the official Snapdragon backend provided by `llama.cpp`.

The following tools are required:
- Docker Desktop
- Git
- CMake
- Python

### Docker Desktop

The official Snapdragon backend recommends using the Snapdragon Toolchain Docker image for Android builds.

Install Docker Desktop before proceeding.

Verification:

```bash
docker --version
```

Expected: `Docker version xx.xx.x`

In [1]:
import subprocess

def run(cmd):
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True
    )

    if result.stdout:
        print(result.stdout.strip())

    if result.stderr:
        print(result.stderr.strip())

In [2]:
run(["git", "--version"])

# or 
git = r"C:\Program Files\Git\bin\git.exe"
# run([git, "--version"])

git version 2.45.2.windows.1


In [3]:
# run(["docker", "--version"])
# or 
docker = r"C:\Program Files\Docker\Docker\resources\bin\docker.exe"
run([docker, "--version"])

Docker version 28.1.1, build 4eba377


In [4]:
# run(["cmake", "--version"])
# or
cmake = r"C:\Program Files\CMake\bin\cmake.exe"
run([cmake, "--version"])

cmake version 4.4.0-rc3

CMake suite maintained and supported by Kitware (kitware.com/cmake).


## Clone llama.cpp

The Snapdragon backend is developed within the official `llama.cpp` repository.

This experiment uses the latest upstream source code to ensure compatibility with the latest Qualcomm backend improvements.

If the repository already exists locally, it will be reused.

In [5]:
from pathlib import Path

repo = Path("llama.cpp")

print(f"Repository exists   : {repo.exists()}")

if repo.exists():
    print(f"Location        : {repo.resolve()}")

Repository exists   : True
Location        : C:\Users\ASUS\Desktop\ModelQuantize\github\NPU\llama.cpp


In [6]:
import subprocess
from pathlib import Path

repo = Path("llama.cpp")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/ggml-org/llama.cpp.git"
        ],
        check=True
    )
else:
    print("Using existing repository.")

Using existing repository.


In [7]:
from pathlib import Path

repo = Path("llama.cpp")

assert repo.exists(), "llama.cpp repository not found."

print("Repository verified.")

Repository verified.


### Validation

The experiment proceeds only after the official `llama.cpp` repository has been cloned successfully.

The next step is to inspect the Snapdragon backend included with the current version of `llama.cpp`.

## Inspect Snapdragon Backend

Before attempting to build, verify that the cloned repository contains the official Snapdragon backend documentation and build configuration.

In [8]:
from pathlib import Path

snapdragon_docs = Path("llama.cpp/docs/backend/snapdragon")

print(f"Exists : {snapdragon_docs.exists()}")
print(f"Path   : {snapdragon_docs.resolve()}")

Exists : True
Path   : C:\Users\ASUS\Desktop\ModelQuantize\github\NPU\llama.cpp\docs\backend\snapdragon


In [9]:
from pathlib import Path

for item in sorted(Path("llama.cpp/docs/backend/snapdragon").iterdir()):
    print(item.name)

CMakeUserPresets.json
developer.md
linux.md
README.md
windows.md


---
## Inspect Snapdragon Documentation

The Snapdragon backend provides platform-specific build instructions and backend implementation details.

Before compiling, inspect the available documentation to understand the required toolchain and build process.

This follows a research-first methodology rather than relying on trial-and-error.

In [10]:
from pathlib import Path

docs = Path("llama.cpp/docs/backend/snapdragon")

for file in [
    "README.md",
    "windows.md",
    "developer.md"
]:
    path = docs / file
    print("=" * 80)
    print(file)
    print("=" * 80)

    with open(path, encoding="utf-8") as f:
        print("".join(f.readlines()[:30]))

README.md
# Snapdragon-based devices

## Setup

### Android

The easiest way to build llama.cpp for a Snapdragon-based Android device is using the toolchain Docker image (see github.com/snapdragon-toolchain).
This image includes Android NDK, OpenCL SDK, Hexagon SDK, CMake, etc.

This method works on Linux, macOS, and Windows. macOS and Windows users should install Docker Desktop.

```
~/src/llama.cpp$ docker run -it -u $(id -u):$(id -g) --volume $(pwd):/workspace --platform linux/amd64 ghcr.io/snapdragon-toolchain/arm64-android:v0.7
[d]/> cd /workspace
```

Note: The rest of the **Android** build process assumes that you're running inside the toolchain container.

### Windows On Snapdragon

Native Windows 11 arm64 builds has the following tools dependencies:
- MS Visual Studio 2026 (Community Edition or Pro)
  - MSVC arm64 standard and runtime libraries
  - UCRT and Driver Kit
- LLVM core libraries and Clang compiler (winget)
- CMake, Git, Python (winget)
- Hexagon SDK Community Editio

## Documentation Findings

The official Snapdragon backend documentation was reviewed before beginning the implementation.

The following observations will guide the remainder of this experiment.

### Key Findings

#### Build Environment

- The recommended build environment is the official Snapdragon Toolchain Docker image.
- The Docker image includes Android NDK, Hexagon SDK, OpenCL SDK, CMake, Python and Clang.

---

#### Build System

- `llama.cpp` provides official Snapdragon CMake presets.
- The backend supports Android ARM64 cross-compilation.

---

#### Supported Backends

The Snapdragon backend currently supports:

- CPU
- Adreno GPU (OpenCL)
- Hexagon NPU

---

#### Packaging

The official workflow packages all generated binaries and shared libraries using:

```bash
cmake --install
```

This produces a deployment-ready directory that can be copied directly to an Android device.

---

#### Device Deployment

Deployment to Android is performed using:

```bash
adb push
```

The generated package contains:

- Executables
- Shared libraries
- Required runtime files

---

#### Hexagon Sessions

The backend supports multiple Hexagon execution sessions.

Examples:

- Small models → 1 HTP
- Approximately 8B models → 2 HTPs
- Approximately 20B models → 4 HTPs

This suggests that larger models can be partitioned across multiple Hexagon execution devices.

### Research Outcome

The literature review answers several questions raised during the planning phase.

| Question | Status |
|----------|--------|
| Snapdragon backend available? | ✓ Yes |
| Official build workflow available? | ✓ Yes |
| Android deployment supported? | ✓ Yes |
| Multi-HTP execution supported? | ✓ Yes |
| Q4_0 demonstrated on Hexagon? | ✓ Yes |

The remaining objective is to verify these capabilities experimentally on the Samsung Galaxy S26 Ultra.

## Build Snapdragon Backend

The official Snapdragon backend uses cross-compilation to generate Android ARM64 binaries.

Rather than manually installing the Android NDK, Hexagon SDK and OpenCL SDK, the recommended approach is to use the official Snapdragon Toolchain Docker image. This provides a reproducible build environment across Windows, Linux and macOS.

The following steps prepare the host environment, launch the Snapdragon toolchain, configure the build system and compile `llama.cpp` for Android.

---

### Host Preparation

#### Step 1: Verify Docker Installation

Verify that Docker Desktop is installed and running.

Docker is required because the Snapdragon Toolchain is distributed as a preconfigured Docker image containing all required build dependencies.

Expected Outcome:

- Docker CLI is available
- Docker daemon is running
- Container execution is supported

In [11]:
run([docker, "info"])

Client:
 Version:    28.1.1
 Context:    desktop-linux
 Debug Mode: true
 Plugins:
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.1.7
    Path:     C:\Program Files\Docker\cli-plugins\docker-ai.exe
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.23.0-desktop.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-buildx.exe
  cloud: Docker Cloud (Docker Inc.)
    Version:  v0.3.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-cloud.exe
  compose: Docker Compose (Docker Inc.)
    Version:  v2.35.1-desktop.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-compose.exe
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.38
    Path:     C:\Program Files\Docker\cli-plugins\docker-debug.exe
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.1.8
    Path:     C:\Program Files\Docker\cli-plugins\docker-desktop.exe
  dev: Docker Dev Environments (Docker Inc.)
    Version:  v0.1.2
    Path:     C:\Program

### Step 2: Download the Snapdragon Toolchain

Download the official Snapdragon Toolchain Docker image.

The image contains:

- Android NDK
- Hexagon SDK
- OpenCL SDK
- Clang
- CMake
- Python

This image only needs to be downloaded once.

In [12]:
run([
    docker,
    "pull",
    "ghcr.io/snapdragon-toolchain/arm64-android:v0.7"
])

v0.7: Pulling from snapdragon-toolchain/arm64-android
f52fdc9690f4: Pulling fs layer
0331f0174c5d: Pulling fs layer
bfa19e54cc1c: Pulling fs layer
f52fdc9690f4: Already exists
0331f0174c5d: Already exists
bfa19e54cc1c: Download complete
bfa19e54cc1c: Pull complete
f52fdc9690f4: Pull complete
0331f0174c5d: Pull complete
Digest: sha256:91714433626f0d94a926538a1e46ec43756c5b8e3262b91b95df1e812940aed1
Status: Downloaded newer image for ghcr.io/snapdragon-toolchain/arm64-android:v0.7
ghcr.io/snapdragon-toolchain/arm64-android:v0.7


#### Step 3: Verify Docker Image

Verify that the Snapdragon Toolchain image has been downloaded successfully.

In [13]:
run([
    docker,
    "images",
    "ghcr.io/snapdragon-toolchain/arm64-android:v0.7"
])

REPOSITORY                                   TAG       IMAGE ID       CREATED       SIZE
ghcr.io/snapdragon-toolchain/arm64-android   v0.7      91714433626f   5 weeks ago   9.97GB


### Launch Snapdragon Toolchain

#### Step 4: Start the Toolchain Container

Launch the official Snapdragon Toolchain container and mount the local `llama.cpp` repository into the container workspace.

Once inside the container, all subsequent build commands will be executed from the mounted workspace.

```bash
docker run -it --rm `
-v "${PWD}:/workspace" `
--platform linux/amd64 `
ghcr.io/snapdragon-toolchain/arm64-android:v0.7
```

Inside the container:
```bash
cd /workspace
```

#### Step 5: Configure the Build

The Snapdragon backend provides predefined CMake presets that configure the Android toolchain, OpenCL backend and Hexagon backend.

Copy the Snapdragon CMake presets into the repository root before configuring the build.

```bash
cp docs/backend/snapdragon/CMakeUserPresets.json .
```

Configure the build using the Snapdragon release preset.

This preset enables:

- Android ARM64 cross-compilation
- Hexagon backend
- OpenCL backend
- Release optimizations

```bash
cmake --preset arm64-android-snapdragon-release -B build-snapdragon
```

#### Step 6: Compile

Compile the Snapdragon-enabled version of `llama.cpp`.

```bash
cmake --build build-snapdragon
```

#### Step 7: Package the Build

Install the generated binaries and libraries into a deployment-ready package.

```bash
cmake --install build-snapdragon --prefix pkg-snapdragon/llama.cpp
```

#### Step 8: Verify Generated Artifacts

After installation, the package should contain:

- `llama-cli`
- `llama-server`
- `llama-bench`
- `libggml-hexagon.so`
- `libggml-opencl.so`
- Supporting runtime libraries

These artifacts are ready to be deployed to the Android device using `adb`.

In [23]:
from pathlib import Path

pkg = Path("llama.cpp/pkg-snapdragon/llama.cpp")

print(f"Package exists : {pkg.exists()}")
print(f"Location       : {pkg.resolve()}")

Package exists : True
Location       : C:\Users\ASUS\Desktop\ModelQuantize\github\NPU\llama.cpp\pkg-snapdragon\llama.cpp


In [24]:
from pathlib import Path

bin_dir = Path("llama.cpp/pkg-snapdragon/llama.cpp/bin")

print("Executables:\n")

for exe in sorted(bin_dir.iterdir()):
    print(exe.name)

Executables:

llama
llama-batched
llama-batched-bench
llama-bench
llama-cli
llama-completion
llama-convert-llama2c-to-ggml
llama-cvector-generator
llama-debug
llama-debug-template-parser
llama-diffusion-cli
llama-embedding
llama-eval-callback
llama-export-lora
llama-finetune
llama-fit-params
llama-gen-docs
llama-gguf
llama-gguf-hash
llama-gguf-split
llama-idle
llama-imatrix
llama-lookahead
llama-lookup
llama-lookup-create
llama-lookup-merge
llama-lookup-stats
llama-mtmd-cli
llama-parallel
llama-passkey
llama-perplexity
llama-quantize
llama-results
llama-retrieval
llama-server
llama-simple
llama-simple-chat
llama-speculative
llama-speculative-simple
llama-template-analysis
llama-tokenize
llama-tts
test-alloc
test-arg-parser
test-autorelease
test-backend-ops
test-backend-sampler
test-barrier
test-chat
test-chat-auto-parser
test-chat-peg-parser
test-chat-template
test-col2im-1d
test-export-graph-ops
test-gbnf-validator
test-gguf
test-grammar-integration
test-grammar-parser
test-jinja
test

In [25]:
from pathlib import Path

lib_dir = Path("llama.cpp/pkg-snapdragon/llama.cpp/lib")

print("Libraries:\n")

for lib in sorted(lib_dir.iterdir()):
    print(lib.name)

Libraries:

cmake
libggml-base.so
libggml-cpu.so
libggml-hexagon.so
libggml-htp-v73.so
libggml-htp-v75.so
libggml-htp-v79.so
libggml-htp-v81.so
libggml-opencl.so
libggml.so
libllama-batched-bench-impl.so
libllama-bench-impl.so
libllama-cli-impl.so
libllama-common.so
libllama-completion-impl.so
libllama-fit-params-impl.so
libllama-perplexity-impl.so
libllama-quantize-impl.so
libllama-server-impl.so
libllama.so
libmtmd.so
pkgconfig


### Validation

The build is considered successful if the deployment package contains:

- `llama-cli`
- `llama-server`
- `llama-bench`
- `libggml-hexagon.so`
- `libggml-opencl.so`
- `libggml.so`

The generated package is now ready to be transferred to the Android device for runtime validation.

## Conclusion

The Snapdragon backend was successfully built using the official Snapdragon Toolchain.

The resulting package contains Android ARM64 executables together with the required Qualcomm runtime libraries.

This confirms that the host development environment is correctly configured for Snapdragon cross-compilation.

The next experiment focuses on deploying the generated package to the Samsung Galaxy S26 Ultra and validating the available execution backends.